<div class = "alert alert-info"> <h2 align = "center"> <b> Cross-asset Macro Overlay </b> </h2> 
<h3 align = center> <font  color = "black"> Backtest<font/> </h3>

</div>

### __Contexte__

Les portefeuilles multi-actifs sont traditionnellement construits sur des fondations quantitatives (corrélations, volatilités, betas). Cependant, une grande partie des mouvements de marché est macro-driven (inflation, taux, croissance, politique monétaire).

Deux sources d’information sont cruciales mais souvent exploitées séparément :

- Marché des options de taux → anticipe les mouvements futurs des banques centrales et l’incertitude (via vol implicite, skew, term structure).

- Flux de nouvelles macroéconomiques → messages textuels des banques centrales, données économiques, événements géopolitiques.

__Le projet vise à fusionner ces deux sources pour générer un overlay dynamique cross-asset, capable de repositionner le portefeuille face aux chocs macro.__


### __Objectifs principaux :__

- Construire un overlay dynamique cross-asset basé sur : Signaux textuels macro (news, communiqués, discours de banques centrales); Options rates (volatilité implicite, skew, convexité).

- Ajuster l’allocation en fonction de régimes de marché identifiés via Markov Switching / State-space.

- Quantifier l’impact des news sur la performance du portefeuille (news-to-portfolio attribution).



### __Backtest__

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error  # For potential additional metrics
from viz_theme import set_professional_style, corporate_palette

# Standard reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

set_professional_style(base_size=12)
palette = corporate_palette()



# Load the model results from previous steps (includes port_returns, optimized_weights, regimes, etc.)
df = pd.read_parquet('../data/model_results.parquet')

# Assume key columns: 'port_returns' (dynamic overlay), obs_vars for assets
obs_vars = ['^GSPC', '^STOXX50E', '^IRX', '^TNX', 'EURUSD=X', 'JPYUSD=X', 'CL=F', 'GC=F']

# Use S&P500 (^GSPC) as benchmark
if 'benchmark_returns' not in df.columns:
    df['benchmark_returns'] = df['^GSPC']

# Historical window: 2018-2025 (full data)
start_date = '2018-01-01'
end_date = '2025-01-01'
df = df.loc[start_date:end_date]



In [ ]:
# 6. Backtesting et Validation

# a) Out-of-sample Rolling Window
# Split: e.g., train 2018-2022, test rolling 2023-2025
train_end = '2022-12-31'
test_start = '2023-01-01'

df_train = df.loc[:train_end]
df_test = df.loc[test_start:]

# Rolling window test: e.g., 252 days (1 year) rolling, rebalance/eval
rolling_returns = []
rolling_bench = []
window_size = 252  # Trading days ~1 year

for i in range(window_size, len(df_test)):
    window = df_test.iloc[i-window_size:i]
    # Portfolio return over window (cumulative)
    cum_port = (1 + window['ret_sl']).cumprod().iloc[-1] - 1
    cum_bench = (1 + window['benchmark_returns']).cumprod().iloc[-1] - 1
    rolling_returns.append(cum_port)
    rolling_bench.append(cum_bench)

# Plot rolling out-of-sample performance
plt.figure(figsize=(12, 6))
plt.plot(df_test.index[window_size:], rolling_returns, label='Dynamic Overlay Rolling Return')
plt.plot(df_test.index[window_size:], rolling_bench, label='S&P500 Benchmark Rolling Return')
plt.title('Out-of-Sample Rolling Window Cumulative Returns (252 days)')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()

In [ ]:
# b) Metrics: Sharpe Ratio, Max Drawdown, Information Ratio
# Full test period
port_ret = df_test['ret_sl']
bench_ret = df_test['benchmark_returns']

# Sharpe Ratio: (mean - rf) / std * sqrt(252); assume rf=0 for simplicity
sharpe_port = port_ret.mean() / port_ret.std() * np.sqrt(252) if port_ret.std() != 0 else 0
sharpe_bench = bench_ret.mean() / bench_ret.std() * np.sqrt(252) if bench_ret.std() != 0 else 0

# Max Drawdown
cum_port = (1 + port_ret).cumprod()
cum_bench = (1 + bench_ret).cumprod()
dd_port = (cum_port / cum_port.cummax() - 1).min()
dd_bench = (cum_bench / cum_bench.cummax() - 1).min()

# Information Ratio: (port_mean - bench_mean) / tracking_error (std of excess)
excess_ret = port_ret - bench_ret
ir = excess_ret.mean() / excess_ret.std() * np.sqrt(252) if excess_ret.std() != 0 else 0

print(f"Backtest Metrics (Test Period {test_start} to {end_date}):")
print(f"Sharpe Ratio - Dynamic: {sharpe_port:.2f}, S&P500 Benchmark: {sharpe_bench:.2f}")
print(f"Max Drawdown - Dynamic: {dd_port:.2%}, S&P500 Benchmark: {dd_bench:.2%}")
print(f"Information Ratio: {ir:.2f}")

# Plot Cumulative Returns for Full Test
plt.figure(figsize=(12, 6))
plt.plot((1 + port_ret).cumprod(), label='Dynamic Overlay')
plt.plot((1 + bench_ret).cumprod(), label='S&P500 Benchmark')
plt.title('Out-of-Sample Cumulative Returns')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.savefig("./output/backtest_cum_returns.png", dpi = 300, bbox_inches = 'tight')

plt.show()



In [ ]:
# c) Stress Test sur Régimes Extrêmes
# Identify stress periods (manual dates; adjust based on data)
stress_periods = {
    'COVID-19': ('2020-02-01', '2020-06-30'),  # Approx COVID crash/recovery
    '2022 Inflation': ('2022-01-01', '2022-12-31')  # Inflation spike period
    # Add more, e.g., 'Other Crisis': ('YYYY-MM-DD', 'YYYY-MM-DD')
}

for name, (start, end) in stress_periods.items():
    stress_df = df.loc[start:end]
    
    # Metrics during stress
    stress_port = stress_df['ret_sl']
    stress_bench = stress_df['benchmark_returns']
    
    sharpe_stress_port = stress_port.mean() / stress_port.std() * np.sqrt(252) if stress_port.std() != 0 else 0
    sharpe_stress_bench = stress_bench.mean() / stress_bench.std() * np.sqrt(252) if stress_bench.std() != 0 else 0
    
    cum_stress_port = (1 + stress_port).cumprod()
    cum_stress_bench = (1 + stress_bench).cumprod()
    dd_stress_port = (cum_stress_port / cum_stress_port.cummax() - 1).min()
    dd_stress_bench = (cum_stress_bench / cum_stress_bench.cummax() - 1).min()
    
    excess_stress = stress_port - stress_bench
    ir_stress = excess_stress.mean() / excess_stress.std() * np.sqrt(252) if excess_stress.std() != 0 else 0
    
    print(f"\nStress Test: {name} ({start} to {end})")
    print(f"Sharpe Ratio - Dynamic: {sharpe_stress_port:.2f}, S&P500 Benchmark: {sharpe_stress_bench:.2f}")
    print(f"Max Drawdown - Dynamic: {dd_stress_port:.2%}, S&P500 Benchmark: {dd_stress_bench:.2%}")
    print(f"Information Ratio: {ir_stress:.2f}")
    
    # Plot for each stress
    plt.figure(figsize=(12, 6))
    plt.plot(cum_stress_port, label='Dynamic Overlay')
    plt.plot(cum_stress_bench, label='S&P500 Benchmark')
    plt.title(f'Cumulative Returns During {name}')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.savefig("./output/backtest_"+name+".png", dpi = 300, bbox_inches = 'tight')

    plt.show()

In [ ]:
# Additional Validation: Regime-Specific Performance
regime_perf = df_test.groupby('regime')[['ret_sl', 'benchmark_returns']].agg({
    'ret_sl': ['mean', 'std'],
    'benchmark_returns': ['mean', 'std']
})
print("\nRegime-Specific Performance (Test Period):")
print(regime_perf)


In [ ]:
# Save backtest results
backtest_summary = {
    'sharpe_dynamic': sharpe_port,
    'sharpe_bench': sharpe_bench,
    'max_dd_dynamic': dd_port,
    'max_dd_bench': dd_bench,
    'info_ratio': ir
}
pd.DataFrame([backtest_summary]).to_csv('./output/backtest_summary.csv', index=False)
print("Backtesting et validation completed. Summary saved to ./output/backtest_summary.csv")